In [51]:
import sys
from pathlib import Path

repo_root = Path(r"C:\Users\Omen\Downloads\my-rag-app").resolve()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

In [52]:
import time
from contextlib import contextmanager


@contextmanager
def timer(name):
    start = time.perf_counter()
    yield
    print(f"{name}: {(time.perf_counter() - start) * 1000:.2f} ms")

In [ ]:
from my_rag_app.constants import DEFAULT_TOP_K_RERANK, DEFAULT_TOP_K_RETRIEVE, QDRANT_COLLECTION, QDRANT_URL
from my_rag_app.core.guardrails.pii import PIIDetector
from my_rag_app.core.guardrails.validation import CitationValidator, InputValidator
from my_rag_app.core.prompting.context_builder import ContextBuilder
from my_rag_app.core.prompting.prompt_builder import PromptBuilder
from my_rag_app.core.qdrant.reranker import CrossEncoderReranker
from my_rag_app.core.qdrant.retriever import HybridRetriever
from my_rag_app.logger import get_logger
from my_rag_app.models.load import LLMClient

logger = get_logger(__name__)

class EmailAssistant:
    """End-to-end query pipeline: retrieve, rerank, build context, and generate an answer."""

    def __init__(self) -> None:
        """Initialise all pipeline components once — reused across queries."""
        self.retriever = HybridRetriever(qdrant_url=QDRANT_URL, collection_name=QDRANT_COLLECTION)
        self.context_builder = ContextBuilder()
        self.prompt_builder = PromptBuilder()
        self.llm = LLMClient()
        self.reranker = CrossEncoderReranker()
        self.input_validator = InputValidator()
        self.citation_validator = CitationValidator()
        self.pii_detector = PIIDetector()

    def ask(
        self,
        query: str,
        chat_history: list[dict[str, str]] | None = None,
    ) -> tuple[str, list[dict]]:
        """Answer a natural-language question, returning (answer, source_payloads).

        chat_history is a list of {"role": "user"/"assistant", "content": "..."} dicts
        representing prior turns — passed to the LLM for conversational context.
        """
        with timer("Total query time"):
            input_result = self.input_validator.validate(query)
            if not input_result.is_valid:
                return input_result.reason, []

        with timer("Retrieval"):
            results = self.retriever.search(query, top_k=DEFAULT_TOP_K_RETRIEVE)
            if not results:
                return "No relevant emails found for this question.", []

        with timer("Reranking"):
            top_results = self.reranker.rerank(query, results, top_k=DEFAULT_TOP_K_RERANK)

        with timer("Thread Expansion"):
            threads = self.retriever.expand_threads(top_results)

        with timer("Context Building"):
            context = self.context_builder.build(top_results, threads)

        with timer("Prompt Builder"):
            messages = self.prompt_builder.build(
            query, context, chat_history=chat_history or []
        )

        with timer("LLM Generation"):
            response = self.llm.generate(messages)

        with timer("PII Detection"):
            self.pii_detector.check(response.content)

        with timer("Citation Validation"):
            citation_result = self.citation_validator.validate(
            response.content, num_context_emails=len(top_results)
        )
        if not citation_result.is_valid:
            return self.citation_validator.fallback_message(), []

        sources = [r["payload"] for r in top_results]
        return response.content, sources


if __name__ == "__main__":
    print("Email Intelligence Assistant — type your question, or 'exit' to quit.\n")
    assistant = EmailAssistant()

    history: list[dict[str, str]] = []
    while True:
        try:
            query = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nExiting.")
            break

        if not query:
            continue
        if query.lower() in ("exit", "quit"):
            print("Exiting.")
            break

        try:
            answer, _ = assistant.ask(query, chat_history=history)
        except Exception as e:
            logger.exception("Query failed | error=%s")
            print(f"\n[Error: {e}]\n")
            continue

        history.append({"role": "user", "content": query})
        history.append({"role": "assistant", "content": answer})
        print(f"\nAssistant: {answer}\n")

Email Intelligence Assistant — type your question, or 'exit' to quit.



[2026-07-03 22:22:30,514] httpx - INFO - HTTP Request: GET http://localhost:6333/collections "HTTP/1.1 200 OK"
[2026-07-03 22:22:30,519] my_rag_app.core.qdrant.retriever - INFO - Loading dense model: BAAI/bge-small-en-v1.5


Total query time: 0.01 ms


[2026-07-03 22:22:30,790] my_rag_app.core.qdrant.retriever - INFO - Loading sparse model: Qdrant/bm25
[2026-07-03 22:22:30,790] httpx - INFO - HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
2026-07-03 22:22:30.796 | WARNING  | fastembed.common.model_management:download_files_from_huggingface:223 - Local file sizes do not match the metadata.
[2026-07-03 22:22:31,108] httpx - INFO - HTTP Request: POST http://localhost:6333/collections/email_knowledge_v1/points/query "HTTP/1.1 200 OK"
[2026-07-03 22:22:31,114] my_rag_app.core.qdrant.retriever - INFO - search query='Fuel price for June 2026 show me all the mail or summarize it' filters=None returned 6 results


Retrieval: 635.72 ms


...

Total query time: 0.01 ms
Retrieval: 88.99 ms


Batches: 100%|██████████| 1/1 [00:00<00:00, 41.70it/s]
[2026-07-03 22:23:18,649] httpx - INFO - HTTP Request: POST http://localhost:6333/collections/email_knowledge_v1/points/scroll "HTTP/1.1 200 OK"
[2026-07-03 22:23:18,655] my_rag_app.core.prompting.context_builder - INFO - Context built | threads=1 emails=3 total_tokens=850/5000
[2026-07-03 22:23:18,657] my_rag_app.core.prompting.prompt_builder - INFO - Prompt built | version=v1 query_len=66 context_len=2632 history_turns=4


Reranking: 222.01 ms
Thread Expansion: 24.95 ms
Context Building: 2.65 ms
Prompt Builder: 1.51 ms


[2026-07-03 22:23:22,350] my_rag_app.models.load - INFO - Generation complete | model=qwen2.5:1.5b input_tokens=2023 output_tokens=144 latency_ms=3688


LLM Generation: 3693.08 ms
PII Detection: 0.03 ms
Citation Validation: 0.01 ms

Assistant: The NOC (Notice of Clearance) for operating cargo flights from Nashik Airport (VAOZ) has been received. The aircraft type is A321/A320F, and the schedule includes:

- SVI3300: OMRK 0030Z – VAOZ 0330Z
Effective: 15 June 2026 to 15 December 2026

- SVI3301: VAOZ 0530Z – OMRK 0830Z
Effective: 15 June 2026 to 15 December 2026

Exiting.
